# Retail BI Capstone — Project 06
**Synthetic demonstration data only.** The available source is a data dictionary, not actual order records. Replace the sample CSV with authorised real transactions before making business claims.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
df = pd.read_csv("retail_sample_transactions_SYNTHETIC.csv")
df.head()

## Data quality checks

In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
quality = {
 "rows": len(df),
 "duplicate_order_ids": int(df.order_id.duplicated().sum()),
 "missing_cells": int(df.isna().sum().sum()),
 "invalid_quantity": int((df.quantity <= 0).sum()),
 "invalid_discount": int((~df.discount_pct.between(0,100)).sum()),
 "invalid_price": int((df.unit_price < 0).sum())
}
quality

## Revenue definitions and KPI scorecard

In [ ]:
df["gross_revenue"] = df.quantity * df.unit_price
df["net_revenue"] = df.gross_revenue * (1-df.discount_pct/100)
df["recognized_revenue"] = np.where(df.payment_status.eq("Paid"),df.net_revenue,0)
paid = df[df.payment_status.eq("Paid")]
pd.Series({
 "Total orders":len(df),
 "Paid orders":len(paid),
 "Paid-order share":len(paid)/len(df),
 "Recognized revenue (INR)":paid.recognized_revenue.sum(),
 "Revenue per paid order (INR)":paid.recognized_revenue.mean(),
 "Average paid-order discount (%)":paid.discount_pct.mean()
})

## Category, segment, city and payment status

In [ ]:
for field in ["category","customer_segment","city"]:
    display(paid.groupby(field).recognized_revenue.sum().sort_values(ascending=False))
display(df.payment_status.value_counts())

## Monthly trend and discount analysis

In [ ]:
monthly = paid.assign(month=paid.order_date.dt.to_period("M").astype(str)).groupby("month").recognized_revenue.sum()
monthly.plot(figsize=(10,4),title="Monthly Recognized Revenue")
plt.ylabel("INR"); plt.xticks(rotation=45); plt.tight_layout(); plt.show()
display(paid.groupby("discount_pct").recognized_revenue.agg(["count","mean","sum"]))

## Recommendations
1. Review Pending/Failed orders daily and reconcile gateway state before customer follow-up.
2. A/B test category merchandising for the leading category; measure conversion and net revenue per visitor.
3. Set discount approval bands and evaluate matched cohorts.

## ROI limitation
True ROI cannot be calculated without cost of goods, contribution margin, marketing spend and implementation cost. Formula: `(incremental contribution profit - implementation cost) / implementation cost × 100`.